# 01 - Block factorization과 attention

**학습 목표**: token을 연속 block으로 나누고, 현재 token이 이전 block 전체와 자기 block을 볼 수 있는 block-causal mask를 만듭니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
# 연속 slice를 사용해 원래 token 순서를 보존한 채 block으로 나눕니다.
def partition(sequence, block_size):
    return [sequence[i:i + block_size] for i in range(0, len(sequence), block_size)]

tokens = list('ABCDEFGH')
blocks = partition(tokens, 3)
print(blocks)
assert blocks == [['A', 'B', 'C'], ['D', 'E', 'F'], ['G', 'H']]

In [ ]:
def block_causal_mask(length, block_size):
    return [
        [(key // block_size) <= (query // block_size) for key in range(length)]
        for query in range(length)
    ]

mask = block_causal_mask(len(tokens), 3)
print('    ' + ' '.join(tokens))
for token, row in zip(tokens, mask):
    print(f'{token}:  ' + ' '.join('1' if x else '.' for x in row))
assert mask[0][2] and not mask[0][3]
assert all(mask[4][j] for j in range(6))

In [ ]:
import math

length = 128
for block_size in (1, 4, 16, 128):
    blocks = math.ceil(length / block_size)
    print(f'block={block_size:3d}: sequential blocks={blocks:3d}, max parallel positions/block={block_size:3d}')


Block size 1은 AR 쪽, 128은 full diffusion 쪽 극단입니다. 큰 block은 병렬 후보가 많지만 likelihood bound와 denoising 난도가 나빠질 수 있습니다.